<a href="https://colab.research.google.com/github/manohar99-1/MYDAILYWORK/blob/main/Image_Captioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q --upgrade transformers torch torchvision pillow streamlit pyngrok

In [10]:
%%writefile app.py
import streamlit as st
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

st.set_page_config(page_title="AI Image Captioning", layout="centered")
st.title("🖼️ AI Image Caption Generator")

device = "cuda" if torch.cuda.is_available() else "cpu"

@st.cache_resource
def load_model():
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
    return processor, model

processor, model = load_model()

uploaded_file = st.file_uploader("Upload an Image", type=["jpg", "png", "jpeg"])

if uploaded_file:
    image = Image.open(uploaded_file).convert("RGB")
    st.image(image, caption="Uploaded Image", use_column_width=True)
    inputs = processor(image, return_tensors="pt").to(device)
    output = model.generate(**inputs, max_length=30)
    caption = processor.decode(output[0], skip_special_tokens=True)
    st.success("Generated Caption:")
    st.markdown(f"<h2 style='text-align: center; color: #4CAF50;'>{caption}</h2>", unsafe_allow_html=True)

Overwriting app.py


In [11]:
from pyngrok import ngrok
import time

!pkill -f streamlit
!pkill -f ngrok

ngrok.set_auth_token("3ARNMlU8fPVorO3y5wy7NUoSwk9_7t1pH18dJaEkQxAiqKTd2")

!streamlit run app.py --server.port 8501 --server.headless true &>/dev/null&

time.sleep(10)

# Verify streamlit is running first
import subprocess
result = subprocess.run(['curl', '-s', '-o', '/dev/null', '-w', '%{http_code}', 'http://localhost:8501'], capture_output=True, text=True)
print("Streamlit status code:", result.stdout)

# Connect with explicit addr
public_url = ngrok.connect(addr="localhost:8501", proto="http")
print(public_url.public_url)

Streamlit status code: 200
https://lirellate-annalee-trilaterally.ngrok-free.dev
